# The Analysis of Token Level Probabilities
Now that I have computed the token-level probabilities, I want to assess their range and the differences (if any) between my chosen programming languages. 

In [1]:
import os
import pandas as pd

In [2]:
python_probs = pd.read_csv('datasets/all_probabilities.csv')

In [4]:
python_probs.head()

,title,line_number,line_text,token_prob,similarity_2,similarity_3,similarity_4,similarity_5
0,Sports Rank,1,def read_input():\n,0.667149,0.971578,0.945252,0.946623,0.945271
1,Sports Rank,2,import sys\n,0.972428,1.000000,0.887545,0.922889,0.940561
2,Sports Rank,3,input = sys.stdin.read\n,0.992203,0.872590,0.833801,0.862679,0.879646
3,Sports Rank,4,data = input().split()\n,0.992855,0.899708,0.814728,0.864639,0.822439
4,Sports Rank,5,index = 0\n,0.939398,0.884853,0.914327,0.931024,0.936494


## Quality Check
Have all lines a computed probability?

In [5]:
python_csv_file = '/datasets/selected_problems_python_14B_no_comments_final.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)

In [7]:
python_probs['line_text'] = python_probs['line_text'].str.strip()

def check_line_probs(df, probs, comment):
    for index, row in df.iterrows():
        lines = row['solution'].split('\n')
        for line in lines:
            line = line.strip()
            matched_rows = probs.loc[(probs['line_text'] == line) & (probs['title'] == row['title'])]
            if matched_rows.empty and not line.startswith(comment) and not line.isspace() and line != '':
                print(row)
                print(f"Line: {line}")

In [8]:
check_line_probs(python_solutions, python_probs, '#')

# Overall Analysis

In [12]:
def get_stats(df, prob):
    overall_min = df[prob].min()
    overall_max = df[prob].max()
    overall_stats = df[prob].agg(['mean', 'median', 'std'])
    return overall_min, overall_max, overall_stats

In [13]:
stats = get_stats(python_probs, 'token_prob')

print(f"Min: {stats[0]}, Max: {stats[1]}")
print("Overall token_prob stats:")
print(stats[2])

Min: 0.3631852009845398, Max: 0.999999642372046
Overall token_prob stats:
mean      0.941532
median    0.982954
std       0.085957
Name: token_prob, dtype: float64


In [16]:
for i in range(2, 6):
    print(f'Similarity {i}')
    stats = get_stats(python_probs, f'similarity_{i}')

    print(f"Min: {stats[0]}, Max: {stats[1]}")
    print("Overall token_prob stats:")
    print(stats[2])

Similarity 2
Min: -0.0454437956213951, Max: 1.000000238418579
Overall token_prob stats:
mean      0.755374
median    0.829256
std       0.239804
Name: similarity_2, dtype: float64
Similarity 3
Min: 0.087679237127304, Max: 1.000000238418579
Overall token_prob stats:
mean      0.754350
median    0.807620
std       0.202689
Name: similarity_3, dtype: float64
Similarity 4
Min: 0.0795100554823875, Max: 1.000000238418579
Overall token_prob stats:
mean      0.749828
median    0.793374
std       0.193026
Name: similarity_4, dtype: float64
Similarity 5
Min: 0.0994904562830925, Max: 1.000000238418579
Overall token_prob stats:
mean      0.752029
median    0.791921
std       0.190702
Name: similarity_5, dtype: float64


In [17]:
x = 0.5
total_python_rows = len(python_probs)
count = (python_probs['token_prob'] < x).sum()
print(f"Number of values lower than {x}: {count} out of {total_python_rows}")

Number of values lower than 0.5: 2 out of 1160


In [19]:
x = 0.5
total_python_rows = len(python_probs)
count = (python_probs['similarity_2'] < x).sum()
print(f"Number of values lower than {x}: {count} out of {total_python_rows}")

Number of values lower than 0.5: 189 out of 1160


# Difficulty-Level Analysis
I also need the selected problems to make a comparison between the probabilties based on the difficulty level.

In [21]:
csv_file = '/datasets/selected_problems.csv'
csv_full_path = os.getcwd()+csv_file
if os.path.exists(csv_full_path):
    selected_problems = pd.read_csv(csv_full_path)
else:
    selected_problems = None
    print("File not found.")

In [22]:
selected_problems.head()

,title,description,difficulty,source
0,Sports Rank,Sports leagues rank their participants (teams ...,Medium,Kattis
1,Solidus Fraction,A fraction written with a solidus (/) can be w...,Medium,Kattis
2,Breakout,A variant of the popular video game breakout u...,Medium,Kattis
3,Passport Dates,Konráð has managed to obtain a big pile of Ice...,Easy,Kattis
4,Weather Forecast,One of the most important things you can do in...,Hard,Kattis


In [23]:
def merge_dfs(df):
    python_probs_merged = pd.merge(df, selected_problems, on='title')
    python_probs_merged = python_probs_merged.drop(['line_number', 'line_text', 'description', 'title'], axis=1)
    return python_probs_merged

In [24]:
python_probs_merged = merge_dfs(python_probs)
python_probs_merged.head()

,token_prob,similarity_2,similarity_3,similarity_4,similarity_5,difficulty,source
0,0.667149,0.971578,0.945252,0.946623,0.945271,Medium,Kattis
1,0.972428,1.000000,0.887545,0.922889,0.940561,Medium,Kattis
2,0.992203,0.872590,0.833801,0.862679,0.879646,Medium,Kattis
3,0.992855,0.899708,0.814728,0.864639,0.822439,Medium,Kattis
4,0.939398,0.884853,0.914327,0.931024,0.936494,Medium,Kattis


In [25]:
range_by_difficulty_python = python_probs_merged.groupby('difficulty')['token_prob'].agg(['min', 'max', 'mean', 'median', 'std'])
print(range_by_difficulty_python)

                 min       max      mean    median       std
difficulty                                                  
Easy        0.591162  0.999999  0.935206  0.982662  0.090595
Hard        0.558246  1.000000  0.939489  0.978605  0.083880
Medium      0.363185  1.000000  0.948133  0.987634  0.085235


In [26]:
range_by_difficulty_sim_2 = python_probs_merged.groupby('difficulty')['similarity_2'].agg(['min', 'max', 'mean', 'median', 'std'])
print(range_by_difficulty_sim_2)

                 min  max      mean    median       std
difficulty                                             
Easy        0.378284  1.0  0.854383  0.887958  0.134429
Hard       -0.008786  1.0  0.724693  0.801929  0.252944
Medium     -0.045444  1.0  0.729990  0.825415  0.258616


In [27]:
range_by_difficulty_sim_3 = python_probs_merged.groupby('difficulty')['similarity_3'].agg(['min', 'max', 'mean', 'median', 'std'])
print(range_by_difficulty_sim_3)

                 min  max      mean    median       std
difficulty                                             
Easy        0.331526  1.0  0.843882  0.868086  0.127155
Hard        0.094367  1.0  0.724755  0.776136  0.201006
Medium      0.087679  1.0  0.733697  0.797797  0.226768
